# Feature Engineering

## Objective

The dataset has been validated and preprocessed.

The objective of this notebook is to transform the processed support ticket data into numerical features suitable for machine learning.

This notebook includes:

- Loading the processed dataset
- Target encoding
- Text vectorization using TF-IDF
- Metadata feature engineering
- Combining text and metadata features
- Train-test split
- Saving feature engineering artifacts

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

from scipy.sparse import hstack

import joblib

## Load Processed Dataset

In [2]:
DATA_PATH = Path("../data/processed/tickets_processed.csv")

df = pd.read_csv(DATA_PATH)

print(df.shape)

df.head()

(16338, 12)


,type,queue,priority,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,text
0,Incident,Technical Support,high,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN,"account disruption dear customer support team,..."
1,Request,Returns and Exchanges,medium,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN,query about smart home system integration feat...
2,Request,Billing and Payments,low,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN,inquiry regarding invoice details dear custome...
3,Problem,Sales and Pre-Sales,medium,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN,question about marketing agency software compa...
4,Request,Technical Support,high,Feature,Product,Documentation,Feedback,NaN,NaN,NaN,NaN,"feature query dear customer support,\n\ni hope..."


## Dataset Overview

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16338 entries, 0 to 16337
Data columns (total 12 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   type      16338 non-null  object
 1   queue     16338 non-null  object
 2   priority  16338 non-null  object
 3   tag_1     16338 non-null  object
 4   tag_2     16332 non-null  object
 5   tag_3     16269 non-null  object
 6   tag_4     14627 non-null  object
 7   tag_5     8416 non-null   object
 8   tag_6     3370 non-null   object
 9   tag_7     1134 non-null   object
 10  tag_8     281 non-null    object
 11  text      16338 non-null  object
dtypes: object(12)
memory usage: 1.5+ MB


## Target Variable

The prediction target is the ticket priority.

Machine learning models require numerical target labels, so the priority classes are encoded using LabelEncoder.

In [4]:
label_encoder = LabelEncoder()

df["priority"] = label_encoder.fit_transform(df["priority"])

df["priority"].head()

0    0
1    2
2    1
3    2
4    0
Name: priority, dtype: int64

In [5]:
print(label_encoder.classes_)

['high' 'low' 'medium']


## Save Target Encoder

The target encoder is saved so that predictions can later be converted back to human-readable priority labels.

In [6]:
MODELS_DIR = Path("../artifacts")

MODELS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    label_encoder,
    MODELS_DIR / "priority_encoder.pkl"
)

['..\\models\\priority_encoder.pkl']

## Verify Target Distribution

In [7]:
df["priority"].value_counts()

priority
2    6618
0    6346
1    3374
Name: count, dtype: int64

In [8]:
print(df.shape)

(16338, 12)


## Metadata Features

In addition to ticket text, the dataset contains structured categorical information.

The following metadata features are included in the machine learning pipeline:

- Ticket Type
- Support Queue
- Tag 1
- Tag 2
- Tag 3
- Tag 4

Tags with excessive missing values are excluded.

In [9]:
metadata_columns = [
    "type",
    "queue",
    "tag_1",
    "tag_2",
    "tag_3",
    "tag_4"
]

df[metadata_columns].head()

,type,queue,tag_1,tag_2,tag_3,tag_4
0,Incident,Technical Support,Account,Disruption,Outage,IT
1,Request,Returns and Exchanges,Product,Feature,Tech Support,NaN
2,Request,Billing and Payments,Billing,Payment,Account,Documentation
3,Problem,Sales and Pre-Sales,Product,Feature,Feedback,Tech Support
4,Request,Technical Support,Feature,Product,Documentation,Feedback


In [10]:
for column in metadata_columns:
    df[column] = df[column].fillna("Unknown")

In [11]:
df[metadata_columns].isna().sum()

type     0
queue    0
tag_1    0
tag_2    0
tag_3    0
tag_4    0
dtype: int64

In [22]:
metadata_features = pd.get_dummies(
    df[metadata_columns],
    dtype=int
)

print(metadata_features.shape)

(16338, 988)


In [23]:
joblib.dump(
    metadata_features.columns.tolist(),
    MODELS_DIR / "metadata_feature_names.pkl"
)

print("Metadata feature names saved successfully.")

Metadata feature names saved successfully.


## One-Hot Encode Metadata

Categorical metadata is converted into numerical features using one-hot encoding.

This representation allows classical machine learning algorithms to use structured ticket information alongside textual features.

In [13]:
print(metadata_features.shape)

(16338, 988)


## Text Vectorization

The ticket text is transformed into numerical features using the TF-IDF (Term Frequency-Inverse Document Frequency) representation.

TF-IDF captures the importance of words within each ticket while reducing the influence of very common words.

In [14]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

In [15]:
text_features = tfidf.fit_transform(df["text"])

print(text_features.shape)

(16338, 5000)


## Save TF-IDF Vectorizer

The trained TF-IDF vectorizer is saved so that the exact same preprocessing can be applied during inference.

In [16]:
joblib.dump(
    tfidf,
    MODELS_DIR / "tfidf_vectorizer.pkl"
)

['..\\models\\tfidf_vectorizer.pkl']

## Combine Text and Metadata Features

The TF-IDF text representation and one-hot encoded metadata are combined into a single feature matrix.

In [17]:
X = hstack([
    text_features,
    metadata_features.values
])

In [18]:
y = df["priority"]

print("Feature Matrix:", X.shape)
print("Target:", y.shape)

Feature Matrix: (16338, 5988)
Target: (16338,)


## Train-Test Split

Split the dataset into training and testing sets while preserving the class distribution using stratified sampling.

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [20]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (13070, 5988)
X_test : (3268, 5988)
y_train: (13070,)
y_test : (3268,)


## Save Training and Testing Data

Persist the train-test split to ensure reproducible model evaluation across all experiments.

In [21]:
joblib.dump(X_train, MODELS_DIR / "X_train.pkl")
joblib.dump(X_test, MODELS_DIR / "X_test.pkl")

joblib.dump(y_train, MODELS_DIR / "y_train.pkl")
joblib.dump(y_test, MODELS_DIR / "y_test.pkl")

print("Training and testing datasets saved successfully.")

Training and testing datasets saved successfully.
